# Phase 6: RLHF — Reward Model + PPO (Later Phase)

**Trigger**: Only use this when DPO improvements plateau across multiple rounds.

**Method**:
1. Train a Tamil **reward model** (Llama 3.2 3B fine-tuned as binary classifier on preference pairs)
2. Run **PPO training** using `trl.PPOTrainer` with the reward model as the scoring function

**Starts from**: `wickkiey/tamil-llama-3.1-8b-dpo-v1`  
**Hardware**: Colab A100 40GB (PPO is memory intensive — holds policy + reference + reward model)  
**Output**: `wickkiey/tamil-llama-3.1-8b-rlhf-v1`

> This is a stub — cells are scaffolded but commented. Uncomment and run when ready.

In [ ]:
# ── Install ────────────────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ["unsloth", "trl>=0.8.6", "datasets", "peft", "accelerate", "bitsandbytes"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Ready.")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
DPO_CHECKPOINT   = "wickkiey/tamil-llama-3.1-8b-dpo-v1"
REWARD_MODEL_BASE = "unsloth/Llama-3.2-3B"   # smaller model as reward model

PREF_PAIRS_FILE  = "../data/preference_pairs.jsonl"  # from Phase 5
MAX_SEQ_LEN      = 512

# Reward model training
RM_EPOCHS        = 2
RM_LR            = 1e-5
RM_BATCH         = 4
RM_OUTPUT        = "outputs/reward_model"
RM_HF_REPO       = "wickkiey/tamil-reward-model-v1"

# PPO training
PPO_LR           = 1e-6
PPO_EPOCHS       = 1
PPO_BATCH        = 8

DPO_POLICY_REPO  = DPO_CHECKPOINT
PPO_OUTPUT       = "outputs/rlhf_v1"
HF_REPO          = "wickkiey/tamil-llama-3.1-8b-rlhf-v1"
HF_TOKEN         = None

print("Config loaded. This is a later-phase notebook — run after DPO plateaus.")

## Step 1: Train Reward Model

Train Llama 3.2 3B to score Tamil responses using our preference pairs from Phase 5.
The reward model learns: `score(chosen) > score(rejected)`.

In [ ]:
# ── Load preference pairs ──────────────────────────────────────────────────────
import json
from datasets import Dataset

pairs = []
with open(PREF_PAIRS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        try:
            pairs.append(json.loads(line))
        except json.JSONDecodeError:
            continue

print(f"Loaded {len(pairs):,} preference pairs")

# Format for reward model: expand each pair into two examples with labels
rm_examples = []
for pair in pairs:
    prompt = pair["prompt"]
    rm_examples.append({"text": prompt + pair["chosen"],   "label": 1})  # chosen = positive
    rm_examples.append({"text": prompt + pair["rejected"], "label": 0})  # rejected = negative

rm_dataset = Dataset.from_list(rm_examples).shuffle(seed=42)
split = rm_dataset.train_test_split(test_size=0.1)
print(f"Reward model dataset: {len(rm_dataset):,} examples")

In [ ]:
# ── Load reward model base ─────────────────────────────────────────────────────
from unsloth import FastLanguageModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

rm_tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_BASE)
if rm_tokenizer.pad_token is None:
    rm_tokenizer.pad_token = rm_tokenizer.eos_token

# Load as sequence classifier (binary: chosen vs rejected)
reward_model = AutoModelForSequenceClassification.from_pretrained(
    REWARD_MODEL_BASE,
    num_labels   = 1,          # single score output
    torch_dtype  = torch.bfloat16,
    device_map   = "auto",
)
print("Reward model base loaded.")

In [ ]:
# ── Train reward model ─────────────────────────────────────────────────────────
from trl import RewardTrainer, RewardConfig

# Tokenize
def tokenize_rm(examples):
    return rm_tokenizer(
        examples["text"], truncation=True,
        max_length=MAX_SEQ_LEN, padding="max_length"
    )

train_rm = split["train"].map(tokenize_rm, batched=True)
eval_rm  = split["test"].map(tokenize_rm, batched=True)

rm_config = RewardConfig(
    output_dir                  = RM_OUTPUT,
    num_train_epochs            = RM_EPOCHS,
    per_device_train_batch_size = RM_BATCH,
    per_device_eval_batch_size  = RM_BATCH,
    learning_rate               = RM_LR,
    logging_steps               = 20,
    eval_strategy               = "steps",
    eval_steps                  = 100,
    save_strategy               = "steps",
    save_steps                  = 100,
    report_to                   = "none",
    max_length                  = MAX_SEQ_LEN,
)

rm_trainer = RewardTrainer(
    model         = reward_model,
    args          = rm_config,
    train_dataset = train_rm,
    eval_dataset  = eval_rm,
    tokenizer     = rm_tokenizer,
)

rm_trainer.train()
print("Reward model training complete.")

reward_model.save_pretrained(RM_OUTPUT)
rm_tokenizer.save_pretrained(RM_OUTPUT)
reward_model.push_to_hub(RM_HF_REPO, token=HF_TOKEN)
print(f"Reward model pushed: https://huggingface.co/{RM_HF_REPO}")

## Step 2: PPO Training

Use the trained reward model to improve the DPO policy via PPO.

> PPO holds policy + reference + reward model simultaneously. Requires ~35–40GB VRAM.

In [ ]:
# ── Load policy model for PPO ──────────────────────────────────────────────────
from unsloth import FastLanguageModel

# Clear GPU memory before loading PPO models
import gc
del reward_model
gc.collect()
torch.cuda.empty_cache()

# Reload reward model (will be used as frozen scorer)
from transformers import AutoModelForSequenceClassification
reward_model = AutoModelForSequenceClassification.from_pretrained(
    RM_OUTPUT, torch_dtype=torch.bfloat16, device_map="auto"
)
reward_model.eval()

# Load policy (DPO model)
policy_model, policy_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = DPO_POLICY_REPO,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
policy_model = FastLanguageModel.get_peft_model(
    policy_model, r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=64, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

print(f"GPU memory after model loads: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ── PPO training loop ──────────────────────────────────────────────────────────
from trl import PPOTrainer, PPOConfig

ppo_config = PPOConfig(
    model_name    = DPO_POLICY_REPO,
    learning_rate = PPO_LR,
    batch_size    = PPO_BATCH,
    mini_batch_size = 2,
    gradient_accumulation_steps = 4,
    optimize_cuda_cache = True,
    log_with      = None,
    seed          = 42,
)

ppo_trainer = PPOTrainer(
    config    = ppo_config,
    model     = policy_model,
    tokenizer = policy_tokenizer,
)

# Reward function: uses reward_model to score responses
def compute_reward(responses: list[str]) -> list[torch.Tensor]:
    inputs = rm_tokenizer(
        responses, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LEN, padding=True
    ).to(reward_model.device)
    with torch.inference_mode():
        scores = reward_model(**inputs).logits.squeeze(-1)
    return [s for s in scores.cpu()]

# Load seed prompts for PPO
ppo_prompts = [p["prompt"] for p in pairs[:2000]]
print(f"PPO trainer ready with {len(ppo_prompts)} seed prompts.")

In [ ]:
# ── Run PPO epochs ─────────────────────────────────────────────────────────────
from tqdm import tqdm
import random

for epoch in range(PPO_EPOCHS):
    random.shuffle(ppo_prompts)
    batches = [ppo_prompts[i:i+PPO_BATCH] for i in range(0, len(ppo_prompts), PPO_BATCH)]

    for batch_prompts in tqdm(batches, desc=f"PPO epoch {epoch+1}"):
        # Tokenize prompts
        query_tensors = [policy_tokenizer(p, return_tensors="pt").input_ids.squeeze(0)
                         for p in batch_prompts]

        # Generate responses from current policy
        with torch.inference_mode():
            response_tensors = ppo_trainer.generate(
                query_tensors,
                max_new_tokens = 150,
                temperature    = 0.7,
                do_sample      = True,
                pad_token_id   = policy_tokenizer.eos_token_id,
            )

        responses = [policy_tokenizer.decode(r, skip_special_tokens=True)
                     for r in response_tensors]

        # Score with reward model
        rewards = compute_reward(responses)

        # PPO step
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

print("PPO training complete.")

In [ ]:
# ── Save RLHF model ────────────────────────────────────────────────────────────
import os
os.makedirs(PPO_OUTPUT, exist_ok=True)
policy_model.save_pretrained(PPO_OUTPUT)
policy_tokenizer.save_pretrained(PPO_OUTPUT)

policy_model.push_to_hub(HF_REPO, token=HF_TOKEN,
    commit_message="RLHF v1 — PPO on DPO checkpoint with Tamil reward model")
policy_tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"Pushed: https://huggingface.co/{HF_REPO}")